# Lekse 18: Sikre AI-agenter med kryptografiske kvitteringer

## Praktisk notatbok

Denne notatboken går gjennom fire øvelser:

1. **Signer din første kvittering** for et agentverktøy-kall og verifiser den.
2. **Endre kvitteringen** og se at verifiseringen feiler.
3. **Bygg en kjede med tre kvitteringer** og bekreft kjedens integritet.
4. **Pakk inn et Microsoft Agent Framework-verktøy-kall** slik at hver handling genererer en kvittering.

Alle kryptografiske primitivaer er importert fra godt vedlikeholdte biblioteker (`pynacl` for Ed25519, `jcs` for RFC 8785 canonical JSON, `hashlib` fra Python standardbiblioteket for SHA-256). Kvitteringslogikken selv er enkel Python som du kan lese og endre.

Kjør cellene i rekkefølge. Hver seksjon er kort og selvstendig.


## Oppsett

Installer de to avhengighetene. Begge har tillatende lisenser (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Hjelpeverktøy

Disse to hjelpefunksjonene håndterer base64url-koding (uten utfylling) og SHA-256 hashing av vilkårlige objekter. De holder resten av notatboken fokusert på kvitteringslogikken selv.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.b64decode(s + padding, altchars=b"-_", validate=True)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Seksjon 1: Signer din første kvittering

Forestill deg at vår agent for **Contoso Travel** nettopp har søkt etter flyreiser fra Sydney til Los Angeles for en kunde. Vi ønsker å registrere gatewayens utsagn om dette verktøysanropet slik at en fremtidig revisor kan verifisere det under en separat festet utstedernøkkel.

### Trinn 1.1: Generer en signeringsnøkkel

I produksjon ville agentens signeringsnøkkel ligge i en hardware-sikkerhetsmodul (HSM), Azure Key Vault eller en tilsvarende beskyttet lagringsplass. For denne leksjonen genererer vi en ny nøkkel i minnet.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0


### Trinn 1.2: Bygg kvitteringsinnholdet

Innholdet inneholder alt vi ønsker at kvitteringen skal bekrefte: hvem som handlet, hvilket verktøy, med hvilke argumenter, hva som kom tilbake, under hvilken policy, og når. Vi hasher argumentene og resultatet i stedet for å inkludere dem direkte for å unngå at kvitteringen lekker sensitivt innhold.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Trinn 1.3: Signer og samle kvitteringen

Tre trinn:

1. Kanoniser nyttelasten ved å bruke JCS slik at to implementasjoner som produserer samme logiske kvittering produserer byte-identiske bytes.
2. Signer de kanoniske bytene direkte med Ed25519 privatnøkkelen. PureEdDSA hasher meldingen internt, så en ekstra forhånds-hash ville endre protokollen.

Signaturen festes deretter til den opprinnelige nyttelasten for å produsere den endelige kvitteringen.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "Eu3MmrOSdGq2DuemkjNSPKfz5xbr18okNeTU11NJu2usnsnKtC-pjq2PZM1Oap-WXdVgYkL4iW6SeWPSZ2M9BQ",
    "public_key": "g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0"
  }
}


### Trinn 1.4: Verifiser kvitteringen

Verifisering reverserer prosessen. Vi fjerner signaturen, beregner de kanoniske bytene på nytt, og sjekker signaturen kun etter at den innebygde offentlige nøkkelen samsvarer med en nøkkel som er forhåndsgodkjent av tillitspartiet.

Etter at godkjente nøkler er forhåndsgodkjent utenfor bånd, krever verifisering ingen servicetilkalling. Kvitteringen alene beviser selvkonsistens, ikke at den innebygde nøkkelen tilhører den påståtte utstederen.


In [7]:
def verify_receipt(receipt: dict, trusted_public_keys: set[str]) -> bool:
    # The signature is a structured object: {"alg", "sig", "public_key"}.
    if not isinstance(receipt, dict):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    try:
        # An inline key proves only self-consistency. Trust is configured out of band.
        public_key = sig_obj.get("public_key")
        signature = sig_obj.get("sig")
        if not isinstance(public_key, str) or not isinstance(signature, str):
            return False
        if public_key not in trusted_public_keys:
            return False
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical_bytes = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(public_key))
        verify_key.verify(canonical_bytes, b64url_decode(signature))
        return True
    except (CryptoError, TypeError, ValueError, KeyError, OverflowError):
        return False

is_valid = verify_receipt(receipt, TRUSTED_PUBLIC_KEYS)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, TRUSTED_PUBLIC_KEYS)}")

Receipt is valid: True
Pre-hashed receipt valid: False


Du burde se `Receipt is valid: True` og `Pre-hashed receipt valid: False`. Den positive saken beviser at den direkte-JCS signeringsstien fungerer; den negative kontroller gjør signatur-omfangsregelen kjørbar i stedet for å la den være som prosa.


## Seksjon 2: Manipulere kvitteringen

Hele poenget med kvitteringer er at de skal være manipulasjonsbeviste. La oss bevise det.

Vi vil endre nøyaktig ett tegn i kvitteringen og se at verifiseringen feiler.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, TRUSTED_PUBLIC_KEYS)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Hva skjedde akkurat nå?

Når vi endret `policy_id`, endret de kanoniske bytene seg. Signaturen (som var over de opprinnelige kanoniske bytene) stemmer ikke lenger. Verifisering returnerer korrekt `False`.

Det er umulig å endre et signert felt og samtidig beholde den opprinnelige signaturen gyldig uten den private nøkkelen. En verifisør må også fastsette den aksepterte offentlige nøkkelen utenfor kvitteringen; å publisere eller innebygge en nøkkel alene etablerer ikke tillit.

Prøv selv: endre `tool_name` eller `agent_id` eller `timestamp` i cellen over og kjør på nytt. Hver endring gir en ugyldig kvittering.


## Seksjon 3: Koble kvitteringer sammen

En enkelt kvittering beskytter én handling. De fleste agenter utfører mange handlinger. For å gjøre hele sekvensen manipulasjonssikker, kobler vi hver kvittering til den forrige ved å inkludere forrige kvitterings hash i den nye kvitteringens data.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Hvis noen fjerner eller omrokerer en kvittering i den presenterte sekvensen, brytes neste ledd. Å oppdage avkorting eller omskriving av etterfølgende tegn under signatarens kontroll krever også en festet gjeldende hode, ekstern kontrollpost, eller autentisert befolkningsforpliktelse.


In [9]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [10]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:d54d1e138bb080dd6bd825d9f015efc74ad6c7f78ee9504d0b68b39d0733b39c
Receipt 2: tool=confirm_booking, prev=sha256:9fbe5dc2ffcffd7d167ef13a744aecd578e474fb536cb7104ddb1a359607f2f6


In [11]:
def verify_chain(chain: list, trusted_public_keys: set[str]) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt, trusted_public_keys)

        fields = receipt if isinstance(receipt, dict) else {}
        chain_ok = False
        try:
            if i == 0:
                chain_ok = "previous_receipt_hash" in fields and fields["previous_receipt_hash"] is None
            elif isinstance(chain[i - 1], dict):
                expected = receipt_hash(chain[i - 1])
                chain_ok = fields.get("previous_receipt_hash") == expected
        except (TypeError, ValueError, OverflowError):
            pass  # An unhashable/malformed predecessor cannot establish a chain link.

        seq_ok = type(fields.get("sequence")) is int and fields["sequence"] == i
        shape_ok = isinstance(fields.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": fields.get("tool_name", "<malformed>"),
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, TRUSTED_PUBLIC_KEYS):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Bryt nå kjeden ved å tukle med kvitteringen i midten og verifiser på nytt. Den tuklede kvitteringen mislykkes i signatursjekken, OG den neste kvitteringen mislykkes i kjedelinksjekken (fordi dens `previous_receipt_hash` ikke lenger samsvarer med den endrede midtre kvitteringens hash).


In [12]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, TRUSTED_PUBLIC_KEYS):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Kvitto 0 bekrefter fortsatt (det ble ikke endret og har ingen forgjenger å være avhengig av). Kvitto 1 mislykkes i signatursjekken fordi vi endret `tool_args_hash`. Kvitto 2 mislykkes i kjede-lenke-sjekken fordi dens `previous_receipt_hash` ble beregnet mot det opprinnelige (nå endrede) kvitto 1.

Selv om en angriper signerer den endrede kvitto 1 på nytt (noe de ikke kan gjøre uten den private nøkkelen), ville kjede-lenke-ustemmelsen i kvitto 2 fortsatt avsløre manipuleringen. For å skjule endringen, måtte angriperen signere hvert kvitto på nytt fra endringspunktet og utover, noe som krever besittelse av den private nøkkelen.


## Seksjon 4: Pakk inn et agentverktøys kall med kvitteringssignering

I en ekte distribusjon vil du ikke at hver agentforfatter skal huske å kalle `make_receipt`. Du vil at kvitteringssignering skal være automatisk for hver verktøysanrop.

Her er det enkleste mønsteret: en innpakningsklasse som tar ethvert kallbart verktøyfunksjon og returnerer en kvitteringsutstedende versjon av det. Dette tilpasses til enhver agentrammeverk, inkludert Microsoft Agent Framework (`agent_framework.foundry`).

Hvis du ikke har et Microsoft Foundry-prosjekt satt opp, demonstrerer den lokale mocken nedenfor fortsatt mønsteret.


In [13]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [14]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, TRUSTED_PUBLIC_KEYS):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integrering med Microsoft Agent Framework

Omslaget `ReceiptedTool` ovenfor er rammeverksuavhengig. For å bruke det inne i en agent bygget med Microsoft Agent Framework, registrer den innpakkede funksjonen som et verktøy. Et skjelett (du ville erstatte mocken med en ekte Microsoft Foundry-verktøyregistrering):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agentrammeverket trenger ikke å vite noe om kvitteringer. Kvitteringssignering er pakket rundt verktøyet, ikke skrudd inn i rammeverket. Slik legger du til opprinnelse til eksisterende agentkode uten å omskrive agenten.


## Oppsummering og utfordring

Du har:

- Generert et Ed25519-nøkkelpar.
- Bygget og signert en kvittering for et agentverktøyskall.
- Verifisert kvitteringen offline under en festet offentlig nøkkel for den som stoler på den.
- Manipulert en kvittering og observert verifiseringsfeil.
- Bygget en hash-kjedet sekvens av tre kvitteringer.
- Manipulert midten av kjeden og observert både signaturfeil og feil i kjedelenken.
- Pakket inn en verktøyfunksjon med automatisk kvitteringssignering.

**Utfordring.** Utvid kvitteringsskjemaet med et `request_id`-felt (en UUID for distribuert sporing). Oppdater `make_receipt` til å inkludere det, og bekreft at kvitteringene fortsatt verifiseres fullt ut. Endre deretter feltet etter signering og bekreft at verifiseringen feiler. Dette tvinger deg til å internalisere hvordan hver byte av den kanoniske koding bidrar til signaturen.

**Viktig grense.** Under en festet nøkkel- og kjede-kontekst beviser kvitteringer signaturtilskrivelse, innholds-integritet og en lenke til den presenterte forgjengeren. De beviser IKKE utførelse, kildesannhet, betrodd tid, fullstendighet, policyvurdering eller korrekthet. Kvitteringer er et fundament. Styring er systemet du bygger oppå.

Les leksjonens README igjen med den grensen i tankene. Den vanligste feilen team gjør med kvitteringer er å anta at "vi har kvitteringer" betyr "vi er styrt." Det gjør det ikke. Kvitteringer gjør agentatferd reviderbar. De gjør den ikke korrekt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
